## Libraries & Modules

In [1]:
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from langchain_core.tools import tool

## Deep Agent - Simple

In [2]:
model = init_chat_model("gpt-5")

In [3]:
@tool
def toy_web_search(query: str) -> str:
    """Toy web search tool (stub)."""
    return (
        "RESULT 1: (stub) LangGraph is a graph-based orchestration framework.\n"
        "RESULT 2: (stub) Deep agents use planning + tools + subagents.\n"
        f"QUERY WAS: {query}"
    )

@tool
def toy_arxiv_search(query: str) -> str:
    """Toy arXiv search tool (stub)."""
    return (
        "PAPER A: (stub) Tool-use agents survey\n"
        "PAPER B: (stub) Planning and reflection in agents\n"
        f"QUERY WAS: {query}"
    )

In [4]:
system_prompt = """You are a helpful research assistant.

For non-trivial requests:
1) Use write_todos to create a short plan.
2) Use tools when useful.
3) Produce a concise final answer.
"""

agent = create_deep_agent(
    model=model,
    tools=[toy_web_search, toy_arxiv_search],
    system_prompt=system_prompt,
)

In [9]:
async def run_stream(user_text: str):
    inputs = {"messages": [{"role": "user", "content": user_text}]}

    async for mode, chunk in agent.astream(
        inputs,
        stream_mode=["updates"],
    ):
        print("\n\n[MODE]", mode)
        print("-" * 40)
        print()
        print("[CHUNK]", repr(chunk))
        # if mode == "messages":
        #     # chunk is (message_chunk, metadata) :contentReference[oaicite:9]{index=9}
        #     msg_chunk, meta = chunk
        #     if getattr(msg_chunk, "content", None):
        #         print(msg_chunk.content, end="", flush=True)

        # elif mode == "updates":
        #     # chunk is typically {node_name: update_dict}
        #     # We’ll look for tool calls / tool results / todo writes inside messages.
        #     for node_name, upd in chunk.items():
        #         msgs = upd.get("messages") if isinstance(upd, dict) else None
        #         if not msgs:
        #             continue

        #         for m in msgs:
        #             # Tool calls often live on AI messages as m.tool_calls (or in additional_kwargs)
        #             tool_calls = getattr(m, "tool_calls", None)
        #             if tool_calls:
        #                 names = [tc.get("name") for tc in tool_calls if isinstance(tc, dict)]
        #                 if any(n in ("write_todos",) for n in names):
        #                     print("\n\n[PLANNING] write_todos called:", tool_calls)
        #                 else:
        #                     print("\n\n[TOOL CALL] ", tool_calls)

        #             # Tool outputs are often ToolMessage instances; content contains results
        #             m_type = getattr(m, "type", None)
        #             if m_type == "tool":
        #                 print("\n\n[TOOL RESULT]", getattr(m, "content", "")[:400])

    print("\n\n[DONE]")


In [10]:
await run_stream("Research LangGraph briefly. Make a short plan first, then use both tools, then answer.")



[MODE] updates
----------------------------------------

[CHUNK] {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan first, then use both tools, then answer.', additional_kwargs={}, response_metadata={}, id='c5b7918e-30a1-4148-bb09-42ab4ba1174e')])}}


[MODE] updates
----------------------------------------

[CHUNK] {'SummarizationMiddleware.before_model': None}


[MODE] updates
----------------------------------------

[CHUNK] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2541, 'prompt_tokens': 4554, 'total_tokens': 7095, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2432, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 4480}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_finger

In [11]:
await run_stream("ok nice!! thanks!!")



[MODE] updates
----------------------------------------

[CHUNK] {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='ok nice!! thanks!!', additional_kwargs={}, response_metadata={}, id='079b9afc-bc78-4286-a12e-93aec5739819')])}}


[MODE] updates
----------------------------------------

[CHUNK] {'SummarizationMiddleware.before_model': None}


[MODE] updates
----------------------------------------

[CHUNK] {'model': {'messages': [AIMessage(content="You're welcome! If you need anything else, just let me know.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 4540, 'total_tokens': 4627, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, '

## Deep Agent - With Sub Agents